# Main figure: return curves, 2 rows x 3 columns

| | left | center | right |
|---|---|---|---|
| **top** | Cheetah-Vel | Hopper-Param | ML10 |
| **bottom** | Ant-Dir | Walker-Param | ML45 |

One run of this notebook renders all six panels (`figures/main/{project}.pdf`) plus the shared legend
(`figures/main/legend.pdf`). Same W&B cache (`rl_results/{project}/raw/`), grid filling, EMA, paper style,
palette and axis rules as the other figure notebooks: x = **Environment Steps** (episodes x max episode
length; nominal for Metaworld, whose episodes end on success, so say so in the caption), ending at the
training budget with a labelled last tick; y = **Avg. Return**, shown on the left column only. Every
panel has the **same plot area**; include the PDFs at their native size (no `width=`).

**Mamba placeholder.** While `PLACEHOLDER = {"Mamba"}`, Mamba is not fetched: a seeded random curve
(the mean of the other memory baselines plus smoothed noise) stands in so the layout and legend are final.
When the Mamba runs finish, check each panel's Mamba template in `PANELS`, set `PLACEHOLDER = set()` and
re-run everything. The notebook prints a warning for every placeholder curve.

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator

try:
    display
except NameError:            # plain-python execution (smoke test); Jupyter defines display()
    display = print


from paper_style import set_paper_style   # shared style: font sizes live in paper_style.py only


set_paper_style()

## Config

`PANELS` lists the six panels in grid order (`row`, `col`). Each method entry is
`(template, h, l)`; the `{seed}` field is filled from the panel's `seeds`, or from
`method_seeds[method]` when a panel overrides them (Cheetah-Vel MATE: s1 is excluded as a bad run, s5
takes its place; Ant-Dir MATE: s5 had a wrong setting, so s0-s4). Settings (user's choice,
2026-09-23): Cheetah-Vel h256_l1; Ant-Dir h512_l1; Walker-Param h512_l2 (GPT-2 h512_l1); Hopper-Param
h256_l2 (GPT-2 h256_l1); ML10 h256_l2 and ML45 h512_l2 (GPT-2 1-layer in both). Markov is swept at 2h (its conditioner width).
MATE projection and Oracle are not shown. `constant` replaces a method by a flat line (Ant-Dir Markov = 200,
the maximum return of a memoryless agent there). Metaworld uses the `_v3` sweep, cut at 200k episodes.

Ant-Dir GPT-2 and the Walker-Param baselines have no plain `_v2` run; their `*_rl2x512_seq32_*_v2`
runs are the same setting (identical `config_rl` to the main `_v2` runs, k = 32 = the baselines' standard).

In [2]:
ENTITY = "mate_research"
METHODS = ["MATE", "GPT-2", "LSTM", "SplAgger", "Mamba", "Markov"]   # legend / draw order (fixed paper order)
STYLE = {                            # fixed paper palette
    "MATE":     dict(color="#E41A1C", ls="-"),   # ours: drawn thicker and on top (EMPHASIS)
    "GPT-2":    dict(color="#009E73", ls="-"),
    "LSTM":     dict(color="#0072B2", ls="-"),
    "SplAgger": dict(color="#AA3377", ls="-"),
    "Mamba":    dict(color="#E69F00", ls="-"),
    "Markov":   dict(color="#999999", ls="--"),
}
SEQ_NAME = {"MATE": "mate", "GPT-2": "gpt", "LSTM": "lstm", "SplAgger": "splagger", "Mamba": "mamba", "Markov": "markov"}
PLACEHOLDER = {"Mamba"}              # methods drawn from seeded random data (not fetched); set() once the runs exist

V2 = "{env}_{model}_h{h}_l{l}_s{seed}_v2"
V3 = "{env}_{model}_h{h}_l{l}_s{seed}_v3"
V4 = "{env}_{model}_h{h}_l{l}_s{seed}_v4"
SEQ32 = "{env}_{model}_h{h}_l{l}_rl2x512_seq32_s{seed}_v2"     # same setting as V2 (see above)
PRE = "{env}_{model}_h{h}_l{l}_rl2x512_s{seed}"                 # pre-v2 sweep, same critic (seeds 0-3)


def methods(h, l, gpt=None, templates=None, mamba=V2, base=V2):
    """Per-method (template, h, l): memory models at (h, l), GPT-2 at `gpt` = (h, l) if given, Markov at 2h."""
    templates = templates or {}
    gh, gl = gpt or (h, l)
    return {
        "MATE":     (templates.get("MATE", base), h, l),
        "GPT-2":    (templates.get("GPT-2", base), gh, gl),
        "LSTM":     (templates.get("LSTM", base), h, l),
        "SplAgger": (templates.get("SplAgger", base), h, l),
        "Mamba":    (mamba, h, l),                    # check the template once the Mamba runs exist
        "Markov":   (templates.get("Markov", base), 2 * h, l),
    }


PANELS = {   # project -> panel; grid order row-major
    "cheetah-vel":  dict(row=0, col=0, env="cheetah", title="Cheetah-Vel", episode_len=200, seeds=range(5),
                         methods=methods(256, 1),
                         method_seeds={"MATE": [0, 2, 3, 4, 5]}),   # s1 had a problem (user, 2026-09-23); s5 keeps n=5
    "hopper-param": dict(row=0, col=1, env="hopper", title="Hopper-Param", episode_len=200, seeds=range(5),
                         methods=methods(256, 2, gpt=(256, 1))),
    "ML10":         dict(row=0, col=2, env="ml10", title="ML10", episode_len=500, seeds=range(6), max_episodes=200000,
                         methods=methods(256, 2, gpt=(256, 1), base=V3, mamba=V4)),
    "ant-dir":      dict(row=1, col=0, env="ant", title="Ant-Dir", episode_len=200, seeds=range(5),
                         methods=methods(512, 1, templates={"GPT-2": SEQ32, "LSTM": PRE}),   # LSTM: user's choice (the _v2 runs ran on RTX 5090, s4 failed)
                         method_seeds={"MATE": [0, 1, 2, 3, 4]},    # s5 excluded: wrong experiment setting (user, 2026-09-23)
                         constant={"Markov": 200.0}),               # a memoryless agent cannot infer the goal direction:
                                                                    # 200 is its maximum return, drawn as a flat reference
    "walker-param": dict(row=1, col=1, env="walker", title="Walker-Param", episode_len=200, seeds=range(5),
                         methods=methods(512, 2, gpt=(512, 1), templates={"GPT-2": SEQ32, "LSTM": SEQ32, "SplAgger": SEQ32})),
    "ML45":         dict(row=1, col=2, env="ml45", title="ML45", episode_len=500, seeds=range(6), max_episodes=200000,
                         methods=methods(512, 2, gpt=(512, 1), base=V3, mamba=V4)),
}

METRIC = "eval/return"
XLABEL = "Environment Steps"
YLABEL = "Avg. Return"               # left column only
EMA_FRACTION = 0.03                  # EMA time constant 1/(1-decay) = 3% of the panel's evaluation points, so
                                     # every env is smoothed over the same fraction of its training (user, 2026-09-23)
EMA_DECAY = 0.97                     # fixed decay, used only when EMA_FRACTION is None (the xlsx workbooks use 0.9)
MISSING_FRACTION_LIMIT = 0.10
MAX_TRAILING_MISSING = 1
LAST_FRACTION = 0.10
HORIZON_EPISODES = None              # None -> each panel's max_episodes, else its last eval point rounded up to 1000

# Panel layout (same helpers as the other figure notebooks): identical plot area for all six panels
TEXT_WIDTH = 5.5
ROW_N, N_YLABEL = 3, 1
ROW_GAP = 0.06
PANEL_H = 1.35
YTICK_RESERVE = "0000"
XTICK_RESERVE_RIGHT = "100M"         # widest last x tick label of the six panels
OUTER_PAD = 0.02
LINE_W = 0.6                         # thinner than the other figures: six methods overlap here (was 0.9)
EMPHASIS, EMPHASIS_SCALE = "MATE", 1.35
BAND_ALPHA = 0.15
ERROR = "ci95"                       # band / bar half-width, with each curve's OWN n of seeds:
                                     # "ci95" = t(0.975, n-1) * std/sqrt(n) (Student-t 95% CI) | "sem" | "std"
DASHES = (3, 1.5)
LEGEND_EDGE = "#d9d9d9"
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)

RESULTS_DIR = Path("rl_results")
FIG_DIR = Path("figures") / "main"   # main-figure files live together here
FORCE_REFRESH = False
UNFINISHED_MAX_AGE_H = 12

## W&B fetch with a local CSV cache (shared with the other notebooks)

In [3]:
def _identity_check(meta, expected):
    problems = [f"{k}={meta.get(k)!r} (expected {v!r})" for k, v in expected.items()
                if meta.get(k) is not None and meta.get(k) != v]
    if problems:
        warnings.warn(f"[{meta['run_name']}] identity mismatch: " + "; ".join(problems))


def fetch_run(project, run_name, force=False):
    """Return (DataFrame[Step, Return], meta dict) or (None, meta) when the run is not found."""
    raw_dir = RESULTS_DIR / project / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    csv_path, meta_path = raw_dir / f"{run_name}.csv", raw_dir / f"{run_name}.meta.json"
    if meta_path.exists() and not force:
        meta = json.loads(meta_path.read_text())
        age_h = (time.time() - meta_path.stat().st_mtime) / 3600
        fresh = UNFINISHED_MAX_AGE_H is None or age_h < UNFINISHED_MAX_AGE_H
        if meta.get("state") == "missing" and fresh:
            return None, meta
        if csv_path.exists() and (meta.get("state") == "finished" or fresh):
            return pd.read_csv(csv_path), meta

    import wandb
    api = wandb.Api(timeout=120)
    runs = list(api.runs(f"{ENTITY}/{project}", filters={"display_name": run_name}))
    if not runs:
        warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{project}")
        meta = {"run_name": run_name, "state": "missing", "checked_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        meta_path.write_text(json.dumps(meta, indent=2))
        if csv_path.exists():
            csv_path.unlink()
        return None, meta
    if len(runs) > 1:                      # prefer a finished run, newest among those
        runs.sort(key=lambda r: (r.state == "finished", str(r.created_at)))
        warnings.warn(f"{len(runs)} runs named {run_name!r}; using {runs[-1].id} (state={runs[-1].state})")
    run = runs[-1]
    cfg = run.config
    seq = cfg.get("config_seq", {}).get("seq_model", {})
    meta = {
        "run_name": run_name, "run_id": run.id, "state": run.state, "created_at": str(run.created_at),
        "eval_interval": cfg.get("config_env", {}).get("eval_interval"),
        "seq_name": seq.get("name"), "is_oracle": seq.get("is_oracle", False),
        "hidden_size": seq.get("hidden_size"), "n_layer": seq.get("n_layer"),
        "project_output": cfg.get("config_seq", {}).get("project_output"),
    }
    rows = [(row["_step"], row[METRIC]) for row in run.scan_history() if row.get(METRIC) is not None]
    df = pd.DataFrame(rows, columns=["Step", "Return"]).sort_values("Step").reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    meta_path.write_text(json.dumps(meta, indent=2))
    return df, meta

## Grid filling and EMA (mirrors the workbook `Methodology` sheet)

In [4]:
def expected_grid(eval_interval, max_episodes):
    return np.arange(eval_interval, int(max_episodes) + 1, eval_interval)


def fill_to_grid(steps, values, grid, missing_fraction_limit=MISSING_FRACTION_LIMIT,
                 max_trailing_missing=MAX_TRAILING_MISSING):
    """Returns (filled values on `grid` or None, status, info dict)."""
    steps = np.asarray(steps, dtype=np.int64)
    values = np.asarray(values, dtype=np.float64)
    on_grid = np.isin(steps, grid)
    n_off_grid = int((~on_grid).sum())
    obs = dict(zip(steps[on_grid], values[on_grid]))          # duplicates: last one wins
    present = np.array([s in obs for s in grid])
    n_expected, n_observed = len(grid), int(present.sum())
    info = dict(eval_points=n_observed, expected_eval_points=n_expected,
                n_missing=n_expected - n_observed, n_off_grid=n_off_grid,
                n_leading=0, n_trailing=0, n_internal=0, exclusion_reason="")
    if n_observed == 0:
        info["exclusion_reason"] = "no evaluations on the grid"
        return None, "excluded", info
    first, last = int(np.argmax(present)), int(len(grid) - 1 - np.argmax(present[::-1]))
    info["n_leading"], info["n_trailing"] = first, n_expected - 1 - last
    info["n_internal"] = info["n_missing"] - info["n_leading"] - info["n_trailing"]
    if info["n_missing"] / n_expected > missing_fraction_limit:
        info["exclusion_reason"] = f"missing fraction {info['n_missing'] / n_expected:.1%} > {missing_fraction_limit:.0%}"
        return None, "excluded", info
    if info["n_trailing"] > max_trailing_missing:
        info["exclusion_reason"] = f"{info['n_trailing']} trailing points missing > {max_trailing_missing}"
        return None, "excluded", info

    xs = grid[present]
    ys = np.array([obs[s] for s in xs])
    filled = np.empty(n_expected)
    filled[first:last + 1] = np.interp(grid[first:last + 1], xs, ys)   # internal linear interpolation
    filled[:first] = ys[0]                                            # leading: repeat first observed
    filled[last + 1:] = ys[-1]                                        # trailing: hold-last
    if info["n_leading"] > 0:
        status = "padded"
    elif info["n_trailing"] > 0:
        status = "tail_extended"
    elif info["n_internal"] > 0:
        status = "interpolated"
    else:
        status = "original"
    return filled, status, info


def ema_decay(n_points):
    """EMA decay whose time constant 1/(1-decay) is EMA_FRACTION of the n_points evaluations of a panel:
    Cheetah (195 evals) ~0.83, Hopper (390) ~0.91, ML (781) ~0.96, Ant/Walker (976) ~0.97."""
    if EMA_FRACTION is None:
        return EMA_DECAY
    return float(np.clip(1.0 - 1.0 / (EMA_FRACTION * n_points), 0.0, 0.999))


def ema(x, decay=EMA_DECAY):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = decay * out[t - 1] + (1.0 - decay) * x[t]
    return out

## Load every panel

Real runs follow the usual rules (a run missing > 10% of its grid, or with > 1 trailing gap, is excluded).
`PLACEHOLDER` methods get a seeded random curve instead and are flagged in the quality table.

In [5]:
import zlib


def placeholder_curve(results, grid, seed):
    """Seeded fake curve: mean of the real memory baselines + smoothed noise (5% of their range)."""
    rng = np.random.default_rng(seed)
    base = [results[m] for m in ("GPT-2", "LSTM", "SplAgger") if m in results]
    mean = np.mean([r["mean"] for r in base], axis=0)
    std = np.mean([r["std"] for r in base], axis=0)
    span = mean.max() - mean.min()
    noise = ema(rng.normal(0, 1, len(grid)).cumsum() / np.sqrt(len(grid)), 0.95) * 0.05 * span
    n = int(np.mean([r["n"] for r in base]))                   # placeholder: typical n of the baselines
    return dict(grid=grid, per_seed=None, mean=mean + noise, std=std, n=n, placeholder=True)


def load_panel(key, panel, force=FORCE_REFRESH):
    raw = {}
    for m, (tmpl, h, l) in panel["methods"].items():
        if m in PLACEHOLDER or m in panel.get("constant", {}):
            continue
        for seed in panel.get("method_seeds", {}).get(m, panel["seeds"]):
            raw[(m, seed)] = fetch_run(key, tmpl.format(env=panel["env"], model=SEQ_NAME[m], h=h, l=l, seed=seed), force=force)
    intervals = {meta["eval_interval"] for df, meta in raw.values() if df is not None}
    assert len(intervals) == 1, f"{key}: eval_interval differs across runs: {intervals}"
    eval_interval = int(intervals.pop())
    max_episodes = panel.get("max_episodes")
    if max_episodes is None:
        last = [int(df["Step"].max()) for df, _ in raw.values() if df is not None and len(df)]
        max_episodes = (max(last) // eval_interval) * eval_interval
    grid = expected_grid(eval_interval, max_episodes)

    results, rows = {}, []
    for m, (tmpl, h, l) in panel["methods"].items():
        if m in PLACEHOLDER or m in panel.get("constant", {}):
            continue
        curves = []
        for seed in panel.get("method_seeds", {}).get(m, panel["seeds"]):
            df, meta = raw[(m, seed)]
            if df is None:
                rows.append(dict(method=m, seed=seed, run_name=meta["run_name"], state=meta["state"],
                                 status="missing", included=False, reason="run not found"))
                continue
            _identity_check(meta, dict(seq_name="markov" if m == "Markov" else SEQ_NAME[m], is_oracle=False,
                                       hidden_size=h, n_layer=l))
            filled, status, info = fill_to_grid(df["Step"].values, df["Return"].values, grid)
            rows.append(dict(method=m, seed=seed, run_name=meta["run_name"], state=meta["state"], status=status,
                             included=filled is not None, reason=info["exclusion_reason"]))
            if filled is not None:
                curves.append(ema(filled, ema_decay(len(grid))))
        if not curves:
            warnings.warn(f"{key} {m}: no included seeds, it will not be drawn")
            continue
        per_seed = np.stack(curves)
        results[m] = dict(grid=grid, per_seed=per_seed, mean=per_seed.mean(0),
                          std=per_seed.std(0, ddof=1) if len(per_seed) > 1 else np.zeros(len(grid)),
                          n=len(per_seed), placeholder=False)
    for m, value in panel.get("constant", {}).items():            # analytic reference line, no seeds, no band
        results[m] = dict(grid=grid, per_seed=None, mean=np.full(len(grid), float(value)), std=np.zeros(len(grid)),
                          n=1, placeholder=False, constant=True)
        rows.append(dict(method=m, seed=None, run_name="(constant)", state="CONSTANT", status=f"= {value}",
                         included=True, reason="constant reference line"))
    for i, m in enumerate(sorted(PLACEHOLDER & set(panel["methods"]))):
        results[m] = placeholder_curve(results, grid, seed=zlib.crc32(f"{key}/{m}".encode()))   # stable across runs
        rows.append(dict(method=m, seed=None, run_name="(placeholder)", state="PLACEHOLDER", status="random",
                         included=True, reason="PLACEHOLDER: random data"))
        warnings.warn(f"{key} {m}: PLACEHOLDER random curve (not real data)")
    return dict(results=results, quality=pd.DataFrame(rows), grid=grid)


loaded = {key: load_panel(key, p) for key, p in PANELS.items()}
quality = pd.concat({k: v["quality"] for k, v in loaded.items()}, names=["panel", None])
summary = quality.groupby(["panel", "method"], sort=False)["included"].sum().unstack()[METHODS]
with pd.option_context("display.max_rows", 300, "display.width", 200):
    display(summary)                                  # included seeds per panel x method
    display(quality[~quality["included"]])            # every excluded / missing run

/tmp/ipykernel_497918/1636451186.py:66: UserWarning: cheetah-vel Mamba: PLACEHOLDER random curve (not real data)
  warnings.warn(f"{key} {m}: PLACEHOLDER random curve (not real data)")
/tmp/ipykernel_497918/1636451186.py:66: UserWarning: hopper-param Mamba: PLACEHOLDER random curve (not real data)
  warnings.warn(f"{key} {m}: PLACEHOLDER random curve (not real data)")
/tmp/ipykernel_497918/1636451186.py:66: UserWarning: ML10 Mamba: PLACEHOLDER random curve (not real data)
  warnings.warn(f"{key} {m}: PLACEHOLDER random curve (not real data)")
wandb: Currently logged in as: himchan00 (piggene00) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/tmp/ipykernel_497918/1636451186.py:66: UserWarning: ant-dir Mamba: PLACEHOLDER random curve (not real data)
  warnings.warn(f"{key} {m}: PLACEHOLDER random curve (not real data)")
/tmp/ipykernel_497918/1636451186.py:66: UserWarning: walker-param Mamba: PLACEHOLDER random curve (not real data)
  warnings.warn(f"{key} {m}: PLAC

method,MATE,GPT-2,LSTM,SplAgger,Mamba,Markov
panel,,,,,,
cheetah-vel,5,5,5,5,1,5
hopper-param,5,5,5,5,1,5
ML10,6,6,6,6,1,6
ant-dir,5,5,4,4,1,1
walker-param,5,5,5,5,1,4
ML45,6,6,6,6,1,6


method  seed                      run_name     state    status  included                         reason
panel                                                                                                                     
ant-dir      14      LSTM   4.0   ant_lstm_h512_l1_rl2x512_s4   missing   missing     False                  run not found
             19  SplAgger   4.0    ant_splagger_h512_l1_s4_v2    failed  excluded     False   missing fraction 26.1% > 10%
walker-param 20    Markov   0.0  walker_markov_h1024_l2_s0_v2  finished  excluded     False  4 trailing points missing > 1

## Panels

Six PDFs with an identical plot area. Line = mean over seeds, band = Student-t 95% confidence interval of the mean
(`ERROR = "ci95"`), each curve with its own number of included seeds n (see the table above); caption:
"mean and 95% confidence interval (Student's t) over n seeds". Only the left column shows the y label; every panel keeps its
x label (the budgets differ). The x axis ends at each panel's training budget.

In [6]:
def _si_formatter(v, _pos):
    for div, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(v) >= div:
            return f"{v / div:g}{suffix}"
    return f"{v:g}"


def budget_tick_candidates(xmax):
    """Tick sets 0..xmax whose LAST tick is exactly xmax (the training budget), best first: a round step
    dividing xmax (densest first), then a round step whose last tick is replaced by / followed by xmax
    (e.g. T-Maze budgets = episodes x (T+1)), then [0, xmax/2, xmax], then [0, xmax]."""
    decade = 10.0 ** np.floor(np.log10(xmax))
    steps = [m * decade for m in (0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5)]
    divides = [abs(xmax / s - round(xmax / s)) < 1e-6 for s in steps]
    for step, exact in zip(steps, divides):
        if exact and 3 <= round(xmax / step) <= 6:
            yield np.linspace(0, xmax, int(round(xmax / step)) + 1)
    for step, exact in zip(steps, divides):
        n = int(np.floor(xmax / step + 1e-9))
        if not exact and 2 <= n <= 6:
            ticks = list(np.arange(n + 1) * step)
            if xmax - ticks[-1] < 0.5 * step:
                ticks[-1] = xmax                               # too close to the budget to label both
            else:
                ticks.append(xmax)
            yield np.array(ticks)
    yield np.array([0, xmax / 2, xmax])
    yield np.array([0, xmax])


def set_budget_xticks(ax, xmax, min_gap_pt=1.0):
    """x axis 0..xmax with the densest candidate tick set whose labels do not collide (call after the
    labels/title are set: it draws the figure to measure the tick labels)."""
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(FuncFormatter(_si_formatter))
    fig = ax.figure
    gap = min_gap_pt * fig.dpi / 72
    for ticks in budget_tick_candidates(xmax):
        ax.set_xticks(ticks)
        fig.canvas.draw()
        boxes = sorted((t.get_window_extent() for t in ax.get_xticklabels() if t.get_text()), key=lambda b: b.x0)
        if all(a.x1 + gap <= b.x0 for a, b in zip(boxes, boxes[1:])):
            break
    ax.set_xlim(0, xmax)
    return ticks


def shade(color, t):
    """Mix `color` toward black by `t` (0 = unchanged): same hue, darker."""
    return matplotlib.colors.to_hex(np.array(matplotlib.colors.to_rgb(color)) * (1.0 - t))


def style_legend_frame(legend):
    """Thin light edge plus a soft drop shadow (stacked offset copies with decreasing alpha; stays vector in PDF)."""
    frame = legend.get_frame()
    frame.set_linewidth(0.4)
    if LEGEND_SHADOW:
        n, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
        effects = [pe.SimplePatchShadow(offset=(size * k / n, -size * k / n), shadow_rgbFace="black",
                                        alpha=alpha / n) for k in range(n, 0, -1)]
        frame.set_path_effects(effects + [pe.Normal()])


def _text_extent(s, size, rotation=0, weight="normal"):
    """(width, height) in inches of `s` at `size` pt with the current rcParams (0 for an empty string)."""
    if not s:
        return 0.0, 0.0
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def panel_margins(show_ylabel=True, show_xlabel=True, title=True):
    """Decoration space (left, right, bottom, top) in inches around the plot area. Tick labels always get the
    room of the widest expected label (YTICK_RESERVE, XTICK_RESERVE_RIGHT), so the plot area sits at the same
    place in every panel; only the y label, x label and title add space, and only where they are shown."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, ytick_h = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    xtick_h = _text_extent("0", rc["xtick.labelsize"])[1]
    left = OUTER_PAD + ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    if show_ylabel:
        left += _text_extent(YLABEL, rc["axes.labelsize"], rotation=90)[0] + rc["axes.labelpad"] * pt
    bottom = OUTER_PAD + xtick_h + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
    if show_xlabel:
        bottom += _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt
    top = OUTER_PAD + ytick_h / 2                              # the top y tick label overhangs the plot
    if title:
        title_h = _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1]
        top = max(top, OUTER_PAD + title_h + rc["axes.titlepad"] * pt)
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    return left, right, bottom, top


def plot_area():
    """(width, height) in inches of the plot area shared by every panel of the figure: ROW_N panels (N_YLABEL
    of them with a y label) plus ROW_GAP gaps fill TEXT_WIDTH, and a panel with title + x label is PANEL_H."""
    l1, r, b, t = panel_margins(show_ylabel=True)
    l0 = panel_margins(show_ylabel=False)[0]
    deco = N_YLABEL * l1 + (ROW_N - N_YLABEL) * l0 + ROW_N * r
    return (TEXT_WIDTH - (ROW_N - 1) * ROW_GAP - deco) / ROW_N, PANEL_H - b - t


def make_panel(show_ylabel=True, show_xlabel=True, title=True):
    """Figure whose plot area is exactly plot_area(); the file is only as large as the decorations it shows."""
    pw, ph = plot_area()
    left, right, bottom, top = panel_margins(show_ylabel, show_xlabel, title)
    w, h = left + pw + right, bottom + ph + top
    fig = plt.figure(figsize=(w, h))
    ax = fig.add_axes([left / w, bottom / h, pw / w, ph / h])
    return fig, ax


def check_panel_fits(fig, ax):
    """Warn when a label is larger than its reserve (it would be clipped at the file edge)."""
    fig.canvas.draw()
    bb, fb = ax.get_tightbbox(fig.canvas.get_renderer()), fig.bbox
    over = {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}
    over = {k: round(v / fig.dpi, 3) for k, v in over.items() if v > 0.5}
    if over:
        warnings.warn(f"labels exceed the panel by {over} in: widen YTICK_RESERVE / XTICK_RESERVE_RIGHT")
    return fig.get_size_inches()


def error_of(r):
    """Half-width of the band, from the curve's OWN number of included seeds n: "ci95" is the Student-t 95%
    confidence interval of the mean, t(0.975, n-1) * std/sqrt(n), so curves with fewer seeds (failed or
    excluded runs) get correspondingly wider bands; "sem" is std/sqrt(n); "std" the sample std (ddof=1)."""
    from scipy.stats import t as student_t
    n = max(int(r["n"]), 1)
    if ERROR == "std":
        return r["std"]
    sem = r["std"] / np.sqrt(n)
    return sem * student_t.ppf(0.975, n - 1) if ERROR == "ci95" and n > 1 else sem


def plot_main_panel(key, panel, data):
    results = data["results"]
    show_ylabel = panel["col"] == 0
    fig, ax = make_panel(show_ylabel=show_ylabel, show_xlabel=True, title=bool(panel.get("title")))
    for z, m in enumerate(METHODS):
        if m not in results:
            continue
        r, st = results[m], STYLE[m]
        is_ref, emph = st["ls"] != "-", m == EMPHASIS
        zorder = 2 + (0 if is_ref else 1) + (1 if emph else 0) + z * 0.01
        x = r["grid"] * panel["episode_len"]
        e = error_of(r)
        ax.fill_between(x, r["mean"] - e, r["mean"] + e, color=st["color"], alpha=BAND_ALPHA,
                        lw=0, zorder=zorder - 1)
        ax.plot(x, r["mean"], color=st["color"], ls=st["ls"], lw=LINE_W * (EMPHASIS_SCALE if emph else 1.0),
                zorder=zorder, dashes=DASHES if is_ref else (None, None))
    episodes = HORIZON_EPISODES or panel.get("max_episodes") or int(np.ceil(data["grid"][-1] / 1000) * 1000)
    budget = episodes * panel["episode_len"]
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.set_title(panel["title"])
    ax.set_xlabel(XLABEL)
    if show_ylabel:
        ax.set_ylabel(YLABEL)
    ax.grid(True, ls="--", alpha=0.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    set_budget_xticks(ax, budget)
    size = check_panel_fits(fig, ax)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / key
    fig.savefig(stem.with_suffix(".pdf"))
    fig.savefig(stem.with_suffix(".png"), dpi=300)
    ph = [m for m in results if results[m].get("placeholder")]
    print(f"saved {stem.with_suffix('.pdf')} ({size[0]:.2f} x {size[1]:.2f} in)" + (f"  PLACEHOLDER: {ph}" if ph else ""))
    return fig, ax


for key, panel in PANELS.items():
    plot_main_panel(key, panel, loaded[key])
    plt.show()

saved figures/main/cheetah-vel.pdf (1.87 x 1.35 in)  PLACEHOLDER: ['Mamba']
saved figures/main/hopper-param.pdf (1.75 x 1.35 in)  PLACEHOLDER: ['Mamba']
saved figures/main/ML10.pdf (1.75 x 1.35 in)  PLACEHOLDER: ['Mamba']
saved figures/main/ant-dir.pdf (1.87 x 1.35 in)  PLACEHOLDER: ['Mamba']
saved figures/main/walker-param.pdf (1.75 x 1.35 in)  PLACEHOLDER: ['Mamba']
saved figures/main/ML45.pdf (1.75 x 1.35 in)  PLACEHOLDER: ['Mamba']


## Shared legend (`figures/main/legend.pdf`, one row, include at native size above the grid)

In [7]:
def save_main_legend(out_stem="legend"):
    handles = []
    for m in METHODS:
        st = STYLE[m]
        is_ref = st["ls"] != "-"
        handles.append(Line2D([], [], color=st["color"], ls=st["ls"], label=m,
                              lw=LINE_W * 1.4 * (EMPHASIS_SCALE if m == EMPHASIS else 1.0),
                              dashes=DASHES if is_ref else (None, None)))
    fig = plt.figure(figsize=(TEXT_WIDTH, 0.3))
    legend = fig.legend(handles=handles, loc="center", ncol=len(handles), frameon=True, fancybox=False,
                        edgecolor=LEGEND_EDGE, facecolor="white", framealpha=1.0, handlelength=2.2,
                        handletextpad=0.6, columnspacing=1.4, borderaxespad=0, borderpad=0.5)
    style_legend_frame(legend)
    stem = FIG_DIR / out_stem
    pad = 0.02 + (LEGEND_SHADOW["size"] / 72 if LEGEND_SHADOW else 0)
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", pad_inches=pad)
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight", pad_inches=pad)
    print(f"saved {stem.with_suffix('.pdf')}")
    plt.show()


save_main_legend()

saved figures/main/legend.pdf


## Final return (mean of the EMA curve over the last 10% of each panel's horizon)

In [8]:
rows = []
for key, data in loaded.items():
    for m, r in data["results"].items():
        sel = r["grid"] >= r["grid"][-1] * (1 - LAST_FRACTION)
        if r.get("constant"):
            rows.append(dict(panel=key, method=m, mean=float(r["mean"][0]), std=0.0, sem=0.0, ci95=0.0, n="constant"))
            continue
        if r["per_seed"] is None:
            rows.append(dict(panel=key, method=m, mean=np.nan, std=np.nan, sem=np.nan, ci95=np.nan, n="placeholder"))
            continue
        fr = r["per_seed"][:, sel].mean(1)
        sd = fr.std(ddof=1) if len(fr) > 1 else 0.0
        from scipy.stats import t as student_t
        ci = sd / np.sqrt(len(fr)) * student_t.ppf(0.975, len(fr) - 1) if len(fr) > 1 else 0.0
        rows.append(dict(panel=key, method=m, mean=fr.mean(), std=sd, sem=sd / np.sqrt(len(fr)), ci95=ci, n=len(fr)))
final = pd.DataFrame(rows)
for col in ("mean", "ci95", "n"):
    display(final.pivot(index="method", columns="panel", values=col).loc[METHODS, list(PANELS)])

panel,cheetah-vel,hopper-param,ML10,ant-dir,walker-param,ML45
method,,,,,,
MATE,-27.805045,531.666673,2498.709880,1113.666250,815.754891,1366.347836
GPT-2,-49.143862,483.044430,2262.560237,1073.619154,679.000984,1396.752815
LSTM,-20.733792,578.208541,1254.202384,1116.840654,802.738104,1074.050995
SplAgger,-21.428318,566.295745,1168.161899,882.611232,748.536940,1279.322744
Mamba,NaN,NaN,NaN,NaN,NaN,NaN
Markov,-163.193030,459.507148,813.211639,200.000000,771.913489,845.645316


panel,cheetah-vel,hopper-param,ML10,ant-dir,walker-param,ML45
method,,,,,,
MATE,2.930360,11.256617,217.429682,22.638689,72.518632,105.278838
GPT-2,6.336863,18.847197,324.981512,20.898501,115.053682,208.705323
LSTM,0.770483,5.856734,51.667221,128.051626,98.551764,112.119717
SplAgger,1.402779,3.843855,149.905071,202.396495,46.385219,309.574545
Mamba,NaN,NaN,NaN,NaN,NaN,NaN
Markov,2.063369,57.325955,67.052739,0.000000,12.345256,34.868426


panel,cheetah-vel,hopper-param,ML10,ant-dir,walker-param,ML45
method,,,,,,
MATE,5,5,6,5,5,6
GPT-2,5,5,6,5,5,6
LSTM,5,5,6,4,5,6
SplAgger,5,5,6,4,5,6
Mamba,placeholder,placeholder,placeholder,placeholder,placeholder,placeholder
Markov,5,5,6,constant,4,6


## LaTeX

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/main/legend.pdf}\\[2pt]
  \includegraphics{figures/main/cheetah-vel.pdf}\hfill
  \includegraphics{figures/main/hopper-param.pdf}\hfill
  \includegraphics{figures/main/ML10.pdf}\\[2pt]
  \includegraphics{figures/main/ant-dir.pdf}\hfill
  \includegraphics{figures/main/walker-param.pdf}\hfill
  \includegraphics{figures/main/ML45.pdf}
  \caption{...}
  \label{fig:main-returns}
\end{figure}
```

All files at native size (no `width=`): the six plot areas are identical, each row is exactly 5.5 in wide.